In [6]:
import requests
import time
import json
from datetime import datetime, timedelta
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, JSON
import base64

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('default')

class GitHubAPITester:
    
    def __init__(self, token=None):
        self.base_url = "https://api.github.com"
        self.headers = {
            'Accept': 'application/vnd.github.v3+json',
            'User-Agent': 'GitHub-Research/1.0'
        }
        
        if token:
            self.headers['Authorization'] = f'token {token}'
            print("авторизованный доступ")
        else:
            print("неавторизованный доступ")
        
        self.request_count = 0
        self.start_time = time.time()
    
    def _request(self, endpoint, params=None):
        url = f"{self.base_url}{endpoint}"
        self.request_count += 1
        
        try:
            response = requests.get(url, headers=self.headers, params=params)
            
            print(f"\n[{self.request_count}] {endpoint}")
            if params:
                print(f"   Параметры: {params}")
            print(f"   Статус: {response.status_code}")
            print(f"   Осталось запросов: {response.headers.get('X-RateLimit-Remaining', 'N/A')}")
            
            response.raise_for_status()
            return response.json()
            
        except requests.exceptions.RequestException as e:
            print(f"ошибка: {e}")
            return None
    
    def search_repos(self, query, page=1, per_page=30):
        return self._request('/search/repositories', {
            'q': query,
            'page': page,
            'per_page': per_page
        })
    
    def get_repo(self, owner, repo):
        return self._request(f'/repos/{owner}/{repo}')
    
    def get_commits(self, owner, repo, page=1, per_page=30):
        return self._request(f'/repos/{owner}/{repo}/commits', {
            'page': page,
            'per_page': per_page
        })
    


github = GitHubAPITester()

# %% [markdown]
# Какие поля возвращает API и как они выглядят

result = github.search_repos("language:c++", per_page=100)

if result and 'items' in result:
    print(f"\n Всего найдено: {result['total_count']:,} репозиториев")
    df = pd.DataFrame(result['items'])
    
    print("\n Доступные поля:")
    for col in df.columns:
        print(f"  • {col}")
    
    print("\nПример данных:")
    display(df[['name', 'full_name', 'size', 'forks_count', 'language']])



In [5]:
# %% [markdown]

# Тестируем разные поисковые запросы
search_queries = [
    "stars:>1000",
    "forks:>500",
    "pushed:>2026-01-01",
    "created:2025-01-01..2025-12-31"
]

for query in search_queries:
    result = github.search_repos(query, per_page=1)
    
    if result:
        total = result['total_count']
        print(f"{query}  {total}")
    
    time.sleep(1)



In [11]:
import base64
import time
import pandas as pd
from datetime import datetime

# %% [markdown]
# ### Задача 1: README с Contributing
# %%
# Соберем две группы проектов:
# 1. Популярные 
# 2. Менее популярные (может не быть Contributing)

#подсчет пулл-реквестов
# Группа A: Очень популярные (5000+ звезд)
def check_contributing(owner, repo):
    readme_url = f"/repos/{owner}/{repo}/readme"
    readme_data = github._request(readme_url)
    
    if readme_data and 'content' in readme_data:
        try:
            content = base64.b64decode(readme_data['content']).decode('utf-8').lower()
            # Проверяем наличие ключевых слов
            keywords = ['contributing', 'how to contribute', 'contribution guide']
            for keyword in keywords:
                if keyword in content:
                    return True
        except:
            return False
    return False

def get_pr_count(owner, repo):
    open_prs = github._request(f"/repos/{owner}/{repo}/pulls", {'state': 'open', 'per_page': 1})
    closed_prs = github._request(f"/repos/{owner}/{repo}/pulls", {'state': 'closed', 'per_page': 1})
    
    open_count = 0
    closed_count = 0

    if open_prs is not None:
        open_count = len(open_prs) if open_prs else 0
    
    if closed_prs is not None:
        closed_count = len(closed_prs) if closed_prs else 0
    return open_count + closed_count

query_a = "stars:>5000 language:python"
result_a = github.search_repos(query_a, per_page=10)

# Группа B: Средней популярности (100-500 звезд)
query_b = "stars:100..500 language:python"
result_b = github.search_repos(query_b, per_page=10)

data_h1 = []

# Анализируем первую группу
print("\nзвезд > 5000:")
for repo in result_a['items']:
    owner, name = repo['full_name'].split('/')
    has_contrib = check_contributing(owner, name)
    pr_count = get_pr_count(owner, name)
    
    data_h1.append({
        'repo': repo['full_name'],
        'stars': repo['stargazers_count'],
        'group': 'популярные (>5000)',
        'has_contributing': has_contrib,
        'pr_count': pr_count
    })
    
    print(f"  {repo['full_name']}: Contributing={has_contrib}, PR={pr_count}")
    time.sleep(2)

# Анализируем вторую группу
print("\nзвезд 100-500:")
for repo in result_b['items']:
    owner, name = repo['full_name'].split('/')
    has_contrib = check_contributing(owner, name)
    pr_count = get_pr_count(owner, name)
    
    data_h1.append({
        'repo': repo['full_name'],
        'stars': repo['stargazers_count'],
        'group': 'средние (100-500)',
        'has_contributing': has_contrib,
        'pr_count': pr_count
    })
    
    print(f"  {repo['full_name']}: Contributing={has_contrib}, PR={pr_count}")
    time.sleep(2)

# Анализ результатов
df_h1 = pd.DataFrame(data_h1)

# Сводная таблица
summary = df_h1.groupby(['group', 'has_contributing'])['pr_count'].agg(['count', 'mean'])
print("\nСводка по группам:")
print(summary)



[188] /search/repositories
   Параметры: {'q': 'stars:>5000 language:python', 'page': 1, 'per_page': 10}
   Статус: 200
   Осталось запросов: 29

[189] /search/repositories
   Параметры: {'q': 'stars:100..500 language:python', 'page': 1, 'per_page': 10}
   Статус: 200
   Осталось запросов: 28

звезд > 5000:

[190] /repos/public-apis/public-apis/readme
   Статус: 200
   Осталось запросов: 4828

[191] /repos/public-apis/public-apis/pulls
   Параметры: {'state': 'open', 'per_page': 1}
   Статус: 200
   Осталось запросов: 4827

[192] /repos/public-apis/public-apis/pulls
   Параметры: {'state': 'closed', 'per_page': 1}
   Статус: 200
   Осталось запросов: 4826
  public-apis/public-apis: Contributing=True, PR=2

[193] /repos/EbookFoundation/free-programming-books/readme
   Статус: 200
   Осталось запросов: 4825

[194] /repos/EbookFoundation/free-programming-books/pulls
   Параметры: {'state': 'open', 'per_page': 1}
   Статус: 200
   Осталось запросов: 4824

[195] /repos/EbookFoundation/free


[248] /repos/makerdiary/python-keyboard/pulls
   Параметры: {'state': 'open', 'per_page': 1}
   Статус: 200
   Осталось запросов: 4772

[249] /repos/makerdiary/python-keyboard/pulls
   Параметры: {'state': 'closed', 'per_page': 1}
   Статус: 200
   Осталось запросов: 4771
  makerdiary/python-keyboard: Contributing=False, PR=2

Сводка по группам:
                                     count      mean
group              has_contributing                 
популярные (>5000) True                 10  2.000000
средние (100-500)  False                 9  1.222222
                   True                  1  2.000000


In [13]:
# %% [markdown]
# ### Задача 2: Чем больше релизов, тем больше звезд

query = "stars:>1000"
result = github.search_repos(query, per_page=30)

data_h2 = []

for i, repo in enumerate(result['items']):
    owner, name = repo['full_name'].split('/')
    
    # Получаем релизы
    releases = github._request(f"/repos/{owner}/{name}/releases", {'per_page': 100})
    release_count = len(releases) if releases else 0
    
    # Возраст проекта
    created = datetime.strptime(repo['created_at'], '%Y-%m-%dT%H:%M:%SZ')
    age_months = (datetime.now() - created).days / 30
    
    # Релизов в месяц
    releases_per_month = release_count / age_months if age_months > 0 else 0
    
    data_h2.append({
        'repo': repo['full_name'],
        'stars': repo['stargazers_count'],
        'total_releases': release_count,
        'age_months': round(age_months, 1),
        'releases_per_month': round(releases_per_month, 2), 
        'has_releases': release_count > 0
    })
    
    status = "ЕСТЬ релизы" if release_count > 0 else "нет релизов"
    print(f"{i+1}. {repo['full_name']}: {status} ({release_count})")
    time.sleep(1)

df_h2 = pd.DataFrame(data_h2)

# Анализируем только проекты с релизами
df_with_releases = df_h2[df_h2['has_releases'] == True].copy()

print(f"\nНайдено проектов с релизами: {len(df_with_releases)} из {len(df_h2)}")

if len(df_with_releases) > 5:
    corr_total = df_with_releases['stars'].corr(df_with_releases['total_releases'])
    print(f"Корреляция кол-ва звезд и кол-ва релизов: {corr_total:.3f}")
    
    corr_per_month = df_with_releases['stars'].corr(df_with_releases['releases_per_month'])
    print(f"Корреляция кол-ва звезд и кол-ва релизов в месяц: {corr_per_month:.3f}")



[271] /search/repositories
   Параметры: {'q': 'stars:>1000', 'page': 1, 'per_page': 30}
   Статус: 200
   Осталось запросов: 28

[272] /repos/codecrafters-io/build-your-own-x/releases
   Параметры: {'per_page': 100}
   Статус: 200
   Осталось запросов: 4750
1. codecrafters-io/build-your-own-x: нет релизов (0)

[273] /repos/sindresorhus/awesome/releases
   Параметры: {'per_page': 100}
   Статус: 200
   Осталось запросов: 4749
2. sindresorhus/awesome: нет релизов (0)

[274] /repos/freeCodeCamp/freeCodeCamp/releases
   Параметры: {'per_page': 100}
   Статус: 200
   Осталось запросов: 4748
3. freeCodeCamp/freeCodeCamp: нет релизов (0)

[275] /repos/public-apis/public-apis/releases
   Параметры: {'per_page': 100}
   Статус: 200
   Осталось запросов: 4747
4. public-apis/public-apis: нет релизов (0)

[276] /repos/EbookFoundation/free-programming-books/releases
   Параметры: {'per_page': 100}
   Статус: 200
   Осталось запросов: 4746
5. EbookFoundation/free-programming-books: нет релизов (0)

In [4]:
# %% [markdown]
# ### Задача 3:есть ссылки, значит будет больше человек отслеживать

# %%
social_patterns = ['twitter.com', 'linkedin.com', 'discord', 'slack', 'youtube.com', 't.me']

def check_social_links(owner, repo):
    readme_url = f"/repos/{owner}/{repo}/readme"
    readme_data = github._request(readme_url)
    
    if readme_data and 'content' in readme_data:
        content = base64.b64decode(readme_data['content']).decode('utf-8').lower()
        found = []
        for pattern in social_patterns:
            if pattern in content:
                found.append(pattern)
        return len(found) > 0, found
    return False, []

# Собираем данные
query = "stars:>1000"
result = github.search_repos(query, per_page=20)

data_h3 = []

for i, repo in enumerate(result['items']):
    owner, name = repo['full_name'].split('/')
    
    has_social, socials_found = check_social_links(owner, name)
    
    full_repo = github.get_repo(owner, name)
    watchers = full_repo.get('watchers_count', 0)
    subscribers = full_repo.get('subscribers_count', 0)
    
    data_h3.append({
        'repo': repo['full_name'],
        'stars': repo['stargazers_count'],
        'has_social': has_social,
        'social_count': len(socials_found),
        'socials': ', '.join(socials_found) if socials_found else 'нет',
        'watchers': watchers,
        'subscribers': subscribers
    })
    
    print(f"{i+1}. {repo['full_name']}: соцсети={has_social} ({len(socials_found)}), watchers={watchers}")
    time.sleep(2)

# Статистический анализ
df_h3 = pd.DataFrame(data_h3)


# 1. Сравнение групп
print("\n1. Сравнение проектов с соцсетями и без:")
grouped = df_h3.groupby('has_social')[['watchers', 'subscribers', 'stars']].mean()
print(grouped)

# 2. Процентное соотношение
has_social_pct = df_h3['has_social'].mean() * 100
print(f"\n2. Проектов с соцсетями: {has_social_pct:.1f}%")




[4] /search/repositories
   Параметры: {'q': 'stars:>1000', 'page': 1, 'per_page': 20}
   Статус: 200
   Осталось запросов: 7

[5] /repos/codecrafters-io/build-your-own-x/readme
   Статус: 200
   Осталось запросов: 56

[6] /repos/codecrafters-io/build-your-own-x
   Статус: 200
   Осталось запросов: 55
1. codecrafters-io/build-your-own-x: соцсети=True (3), watchers=471646

[7] /repos/sindresorhus/awesome/readme
   Статус: 200
   Осталось запросов: 54

[8] /repos/sindresorhus/awesome
   Статус: 200
   Осталось запросов: 53
2. sindresorhus/awesome: соцсети=True (3), watchers=442166

[9] /repos/freeCodeCamp/freeCodeCamp/readme
   Статус: 200
   Осталось запросов: 52

[10] /repos/freeCodeCamp/freeCodeCamp
   Статус: 200
   Осталось запросов: 51
3. freeCodeCamp/freeCodeCamp: соцсети=True (2), watchers=437697

[11] /repos/public-apis/public-apis/readme
   Статус: 200
   Осталось запросов: 50

[12] /repos/public-apis/public-apis
   Статус: 200
   Осталось запросов: 49
4. public-apis/public-ap

In [26]:
# %% [markdown]
# ### Задача 7: Репозитории с permissive-лицензиями (MIT, Apache-2.0) имеют больше stars и forks, чем с другими или без нее

# %%


# Собираем проекты с разными лицензиями
query = "stars:>1000"
result = github.search_repos(query, per_page=50)

data_h7 = []

for i, repo in enumerate(result['items']):
    owner, name = repo['full_name'].split('/')
    
    # Текущая лицензия
    repo_info = github.get_repo(owner, name)
    current_license = repo_info['license']['key'] if repo_info['license'] else None
    
    permissive_licenses = ['mit', 'apache-2.0', 'bsd-3-clause', 'bsd-2-clause']
    if repo['license']:
        license_key = repo_info['license']['key']
        license_name = repo_info['license']['name']
    else:
        license_key = None
        license_name = None
        
    if license_key in permissive_licenses:
        license_type = "permissive"
    elif license_key is None:
        license_type = "no license"
    else:
        license_type = "other"
        
    data_h7.append({
        'repo': repo['full_name'],
        'license_key': license_key,
        'license_name': license_name,
        'license_type': license_type,
        'stars': repo_info['stargazers_count'],
        'forks': repo_info['forks_count'],
        'age_years': round((datetime.now() - datetime.strptime(repo['created_at'], '%Y-%m-%dT%H:%M:%SZ')).days / 365, 1)
    })
    
    print(f"{i+1}. {repo['full_name']}: {license_type} ({license_key}), звезд={repo_info['stargazers_count']}, форков={repo_info['forks_count']}")
    time.sleep(1)

df_h7 = pd.DataFrame(data_h7)

print("\n1. Распеределение проектов по типам лицензий")
license_dist = df_h7['license_type'].value_counts()
for lic_type, count in license_dist.items():
    print(f"{lic_type}: {count} проектов ({count/len(df_h7)*100:.1f}%)")

# 2. Сравнение популярности по типам лицензий
print("\n2.Сравнение популярности оп типам лицензий")

# Группировка по типу лицензии
group_stats = df_h7.groupby('license_type').agg({
    'stars': ['count', 'mean', 'median', 'min', 'max'],
    'forks': ['count', 'mean', 'median']
}).round(0)

print("\nСреднее количество звезд по типам лицензий:")
print(group_stats['stars'])

print("\nСреднее количество форков по типам лицензий:")
print(group_stats['forks'])

        


[301] /search/repositories
   Параметры: {'q': 'stars:>1000', 'page': 1, 'per_page': 50}
   Статус: 200
   Осталось запросов: 29

[302] /repos/codecrafters-io/build-your-own-x
   Статус: 200
   Осталось запросов: 4947
1. codecrafters-io/build-your-own-x: no license (None), звезд=470487, форков=44103

[303] /repos/sindresorhus/awesome
   Статус: 200
   Осталось запросов: 4946
2. sindresorhus/awesome: other (cc0-1.0), звезд=441805, форков=33354

[304] /repos/freeCodeCamp/freeCodeCamp
   Статус: 200
   Осталось запросов: 4945
3. freeCodeCamp/freeCodeCamp: permissive (bsd-3-clause), звезд=437672, форков=43492

[305] /repos/public-apis/public-apis
   Статус: 200
   Осталось запросов: 4944
4. public-apis/public-apis: permissive (mit), звезд=402583, форков=43278

[306] /repos/EbookFoundation/free-programming-books
   Статус: 200
   Осталось запросов: 4943
5. EbookFoundation/free-programming-books: other (cc-by-4.0), звезд=383491, форков=66001

[307] /repos/kamranahmedse/developer-roadmap
   

In [14]:
# %% [markdown]
# ### Задача 4: Анализ сезонности и самых активных месяцев

# %%
from calendar import monthrange

def get_season(month):
    return {12: 'Зима', 1: 'Зима', 2: 'Зима',
            3: 'Весна', 4: 'Весна', 5: 'Весна',
            6: 'Лето', 7: 'Лето', 8: 'Лето',
            9: 'Осень', 10: 'Осень', 11: 'Осень'}.get(month, 'Неизвестно')

def get_last_day(year, month):
    return monthrange(year, month)[1]

months_data = []
years = [2024, 2025, 2026]
current_month = datetime.now().month
current_year = datetime.now().year

for year in years:
    for month in range(1, 13):
        if year == current_year and month > current_month:
            continue
        last_day = get_last_day(year, month)
        month_str = f"{year}-{month:02d}"
        query = f"created:{year}-{month:02d}-01..{year}-{month:02d}-{last_day}"
        
        print(f"\n{month_str}: запрос {query}")
        result = github.search_repos(query, per_page=1)
        
        if result:
            total = result['total_count']
            months_data.append({
                'year': year,
                'month': month,
                'month_name': month_str,
                'season': get_season(month),
                'total': total
            })
            print(f"Создано репозиториев: {total:,}")
        else:
            print("Ошибка получения данных")
        
        time.sleep(1)

df = pd.DataFrame(months_data)

monthly_avg = df.groupby('month')['total'].mean().round(0)
months_ru = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 
             'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']

for month in range(1, 13):
    if month in monthly_avg.index:
        print(f"{months_ru[month-1]}: {monthly_avg[month]:,.0f}")
    else:
        print(f"{months_ru[month-1]}: нет данных")


month_ranking = monthly_avg.sort_values(ascending=False)
for i, (month, avg) in enumerate(month_ranking.items(), 1):
    print(f"  {i}. {months_ru[month-1]}: {avg:,.0f}")


season_stats = df.groupby('season')['total'].agg(['mean', 'std', 'count']).round(0)
print(season_stats)


summer_data = df[df['season'] == 'Лето']
autumn_data = df[df['season'] == 'Осень']

summer_avg = summer_data['total'].mean()
autumn_avg = autumn_data['total'].mean()


print(f"Среднее летом: {summer_avg:,.0f}")
print(f"Среднее осенью: {autumn_avg:,.0f}")


Весна 2025-03:

[302] /search/repositories
   Параметры: {'q': 'created:2025-03-01..2025-03-31', 'page': 1, 'per_page': 1}
   Статус: 200
   Осталось запросов: 29
   Создано репозиториев: 4,759,299

Лето 2025-07:

[303] /search/repositories
   Параметры: {'q': 'created:2025-07-01..2025-07-31', 'page': 1, 'per_page': 1}
   Статус: 200
   Осталось запросов: 28
   Создано репозиториев: 5,247,256

Осень 2025-10:

[304] /search/repositories
   Параметры: {'q': 'created:2025-10-01..2025-10-31', 'page': 1, 'per_page': 1}
   Статус: 200
   Осталось запросов: 27
   Создано репозиториев: 5,811,275

Зима 2025-12:

[305] /search/repositories
   Параметры: {'q': 'created:2025-12-01..2025-12-31', 'page': 1, 'per_page': 1}
   Статус: 200
   Осталось запросов: 26
   Создано репозиториев: 5,666,783

1. Данные по месяцам:
  month season   total
2025-03  Весна 4759299
2025-07   Лето 5247256
2025-10  Осень 5811275
2025-12   Зима 5666783

2. Сравнение активности летом и осенью:
Среднее летом: 5247256 репо

In [27]:
# %% [markdown]
# ### Задача 8-9: Анализ популярности в зависимости от мультиплатформенности

def project_languages(owner, repo):
    langs_url = f"/repos/{owner}/{repo}/languages"
    langs = github._request(langs_url)
    
    if not langs:
        return None, 0, []
    
    total_bytes = sum(langs.values())
    lang_percent = {}
    
    for lang, bytes_count in langs.items():
        percentage = (bytes_count / total_bytes) * 100
        lang_percent[lang] = round(percentage, 1)
        
    sorted_langs = sorted(lang_percent.items(), key=lambda x: x[1], reverse=True)
    major_langs = [lang for lang, pct in sorted_langs if pct > 20]
    is_multiplatform = len(major_langs) >= 2
    
    return is_multiplatform, len(langs), sorted_langs


query = "stars:>1000"
result = github.search_repos(query, per_page=50)    

data = []

for i, repo in enumerate(result['items']):
    owner, name = repo['full_name'].split('/')
    print(f"\n{i+1}. Анализ {repo['full_name']}...")
    
    repo_info = github.get_repo(owner, name)
    
    is_multiplatform, lang_count, lang_details = project_languages(owner, name)
    
    if is_multiplatform is not None:
        watchers = repo_info.get('watchers_count', 0)
        subscribers = repo_info.get('subscribers_count', 0)
        
        data.append({
            'repo': repo['full_name'],
            'is_multiplatform': is_multiplatform,
            'lang_count': lang_count,
            'main_lang': lang_details[0][0] if lang_details else None,
            'main_lang_pct': lang_details[0][1] if lang_details else 0,
            'second_lang': lang_details[1][0] if len(lang_details) > 1 else None,
            'second_lang_pct': lang_details[1][1] if len(lang_details) > 1 else 0,
            'stars': repo_info['stargazers_count'],
            'forks': repo_info['forks_count'],
            'watchers': watchers,
            'subscribers': subscribers
        })
        
        status = "Мультиплатформенный" if is_multiplatform else "Не мультиплатформенный"
        main = f"{lang_details[0][0]} ({lang_details[0][1]}%)"
        second = f"{lang_details[1][0]} ({lang_details[1][1]}%)" if len(lang_details) > 1 else "нет"
        print(f"   {status}")
        print(f"   Языки: основной={main}, второй={second}")
        print(f"   Звезд={repo_info['stargazers_count']}, Форков={repo_info['forks_count']}")
    
    time.sleep(2)

df = pd.DataFrame(data)

print("\n1. Общая Информация:")
print(f"Всего проектов: {len(df)}")
print(f"Мультиплатформенных: {df['is_multiplatform'].sum()} ({df['is_multiplatform'].mean()*100:.1f}%)")
print(f"Не мультиплатформенных: {len(df) - df['is_multiplatform'].sum()} ({(1-df['is_multiplatform'].mean())*100:.1f}%)")

# 2. Сравнение метрик
print("\n2. СРАВНЕНИЕ МЕТРИК:")

multi = df[df['is_multiplatform'] == True]
single = df[df['is_multiplatform'] == False]

print("\nstars:")
print(f"Мультиплатформенные: среднее={multi['stars'].mean():.0f}, медиана={multi['stars'].median():.0f}")
print(f"Одноязычные: среднее={single['stars'].mean():.0f}, медиана={single['stars'].median():.0f}")
print(f"Разница: {((multi['stars'].mean()/single['stars'].mean())-1)*100:.1f}%")

print("\nforks:")
print(f"Мультиплатформенные: среднее={multi['forks'].mean():.0f}, медиана={multi['forks'].median():.0f}")
print(f"Одноязычные: среднее={single['forks'].mean():.0f}, медиана={single['forks'].median():.0f}")
print(f"Разница: {((multi['forks'].mean()/single['forks'].mean())-1)*100:.1f}%")

print("\nWATCHERS:")
print(f"Мультиплатформенные: среднее={multi['watchers'].mean():.0f}, медиана={multi['watchers'].median():.0f}")
print(f"Одноязычные: среднее={single['watchers'].mean():.0f}, медиана={single['watchers'].median():.0f}")
print(f"Разница: {((multi['watchers'].mean()/single['watchers'].mean())-1)*100:.1f}%")
  
    


[352] /search/repositories
   Параметры: {'q': 'stars:>1000', 'page': 1, 'per_page': 50}
   Статус: 200
   Осталось запросов: 29

1. Анализ codecrafters-io/build-your-own-x...

[353] /repos/codecrafters-io/build-your-own-x
   Статус: 200
   Осталось запросов: 4897

[354] /repos/codecrafters-io/build-your-own-x/languages
   Статус: 200
   Осталось запросов: 4948
   Не мультиплатформенный
   Языки: основной=Markdown (100.0%), второй=нет
   Звезд=470489, Форков=44103

2. Анализ sindresorhus/awesome...

[355] /repos/sindresorhus/awesome
   Статус: 200
   Осталось запросов: 4896

[356] /repos/sindresorhus/awesome/languages
   Статус: 200
   Осталось запросов: 4947

3. Анализ freeCodeCamp/freeCodeCamp...

[357] /repos/freeCodeCamp/freeCodeCamp
   Статус: 200
   Осталось запросов: 4895

[358] /repos/freeCodeCamp/freeCodeCamp/languages
   Статус: 200
   Осталось запросов: 4946
   Не мультиплатформенный
   Языки: основной=TypeScript (76.5%), второй=JavaScript (17.9%)
   Звезд=437672, Форков=43


29. Анализ flutter/flutter...

[409] /repos/flutter/flutter
   Статус: 200
   Осталось запросов: 4869

[410] /repos/flutter/flutter/languages
   Статус: 200
   Осталось запросов: 4920
   Не мультиплатформенный
   Языки: основной=Dart (74.7%), второй=C++ (16.4%)
   Звезд=175399, Форков=30070

30. Анализ twbs/bootstrap...

[411] /repos/twbs/bootstrap
   Статус: 200
   Осталось запросов: 4868

[412] /repos/twbs/bootstrap/languages
   Статус: 200
   Осталось запросов: 4919
   Мультиплатформенный
   Языки: основной=MDX (28.9%), второй=JavaScript (25.8%)
   Звезд=174004, Форков=79070

31. Анализ github/gitignore...

[413] /repos/github/gitignore
   Статус: 200
   Осталось запросов: 4867

[414] /repos/github/gitignore/languages
   Статус: 200
   Осталось запросов: 4918

32. Анализ massgravel/Microsoft-Activation-Scripts...

[415] /repos/massgravel/Microsoft-Activation-Scripts
   Статус: 200
   Осталось запросов: 4866

[416] /repos/massgravel/Microsoft-Activation-Scripts/languages
   Статус: 

In [7]:
# %% [markdown]
# ### Задача 10: Зависимость срока жизни проекта от количества форков

# %%
def lifespan(owner, repo):
    
    repo_info = github.get_repo(owner, repo)
    created_at = datetime.strptime(repo_info['created_at'], '%Y-%m-%dT%H:%M:%SZ')
    
    commits_url = f"/repos/{owner}/{repo}/commits"
    last_commits = github._request(commits_url, {
        'per_page': 1,
        'sort': 'created',
        'direction': 'desc'
    })
    
    if not last_commits:
        return None
    
    try:
        last_commit_date = datetime.strptime(last_commits[0]['commit']['author']['date'], '%Y-%m-%dT%H:%M:%SZ')
        
        lifetime = (last_commit_date - created_at).days
        return max(lifetime, 1)  # минимум 1 день
    except:
        return None


query = "stars:>1000"
result = github.search_repos(query, per_page=20) 
data = []

for i, repo in enumerate(result['items']):
    owner, name = repo['full_name'].split('/')
    print(f"\n{i+1}. Анализ {repo['full_name']}...")
    
    repo_info = github.get_repo(owner, name)
    
    lifetime_days = lifespan(owner, name)
    
    if lifetime_days:
        lifetime_years = round(lifetime_days / 365, 1)
        
        forks = repo_info['forks_count']
        stars = repo_info['stargazers_count']
        
        data.append({
            'repo': repo['full_name'],
            'lifetime_days': lifetime_days,
            'lifetime_years': lifetime_years,
            'forks': forks,
            'stars': stars,
            'created': repo['created_at'][:10],
            'updated': repo['updated_at'][:10]
        })
        
        print(f"   Срок жизни: {lifetime_years} лет ({lifetime_days} дней)")
        print(f"   Форки: {forks}, Звезды: {stars}")
    
    time.sleep(2)  

df = pd.DataFrame(data)


print(f"Всего проектов: {len(df)}")
print(f"Средний срок жизни: {df['lifetime_years'].mean():.1f} лет")
print(f"Минимальный срок: {df['lifetime_years'].min():.1f} лет")
print(f"Максимальный срок: {df['lifetime_years'].max():.1f} лет")
print(f"Среднее количество форков: {df['forks'].mean():.0f}")
print(f"Среднее количество звезд: {df['stars'].mean():.0f}")

# Корреляция срока жизни и форков
corr_forks = df['lifetime_years'].corr(df['forks'])
print(f"Корреляция срока жизни и количества форков: {corr_forks:.3f}")

# Корреляция срока жизни и звезд
corr_stars = df['lifetime_years'].corr(df['stars'])
print(f"Корреляция срока жизни и количества звезд: {corr_stars:.3f}")


In [ ]:
# ### Задача 11: Зависимость количества contributors от соотношения открытых/закрытых issues
# %%
import requests
import time
import pandas as pd
import re

def get_contributors_count(owner, repo):
    url = f"https://api.github.com/repos/{owner}/{repo}/contributors"
    response = requests.get(url, headers=github.headers, 
                           params={'per_page': 1, 'anon': 'true'})
    
    if 'Link' in response.headers:
        links = response.headers['Link']
        if 'rel="last"' in links:
            import re
            match = re.search(r'page=(\d+)>; rel="last"', links)
            if match:
                return int(match.group(1))
    
    contributors = github._request(f"/repos/{owner}/{repo}/contributors", 
                                  {'per_page': 100, 'anon': 'true'})
    if contributors:
        if len(contributors) == 100:
            print(f"Контрибьюторов >=100, точное количество неизвестно")
            return 100 
        return len(contributors)
    return 0

def get_issues_count(owner, repo, state):
    if state == 'open':
        query = f"repo:{owner}/{repo} type:issue state:open"
    else:
        query = f"repo:{owner}/{repo} type:issue state:closed"
    
    # Прямой запрос к search/issues
    search_result = github._request('/search/issues', {
        'q': query,
        'per_page': 1
    })
    
    if search_result and 'total_count' in search_result:
        return search_result['total_count']
    
    return 0

def get_issues_stats(owner, repo):
    open_count = get_issues_count(owner, repo, 'open')
    closed_count = get_issues_count(owner, repo, 'closed')
    
    if closed_count > 0:
        issue_ratio = open_count / closed_count
    else:
        issue_ratio = float('inf') if open_count > 0 else 0
    
    return {
        'open_count': open_count,
        'closed_count': closed_count,
        'total_count': open_count + closed_count,
        'ratio': issue_ratio
    }

query = "stars:>1000"
result = github.search_repos(query, per_page=10)

if not result or 'items' not in result:
    print("Ошибка получения списка репозиториев")
else:
    data = []
    
    for i, repo in enumerate(result['items']):
        owner, name = repo['full_name'].split('/')
        print(f"\n{i+1}. Анализ {repo['full_name']}...")
    
        contributors_count = get_contributors_count(owner, name)
        print(f"Всего контрибьюторов: {contributors_count}")
    
        open_count = get_issues_count(owner, name, 'open')
        closed_count = get_issues_count(owner, name, 'closed')
    
        print(f"Открытых задач: {open_count}")
        print(f"Закрытых задач: {closed_count}")
    
        if closed_count > 0:
            ratio = open_count / closed_count
        else:
            ratio = float('inf') if open_count > 0 else 0
    
        print(f"Соотношение: {ratio:.3f}")
    
        data.append({
            'repo': repo['full_name'],
            'contributors': contributors_count,
            'open_issues': open_count,
            'closed_issues': closed_count,
            'issue_ratio': ratio
        })
    
        time.sleep(2)
    
    df = pd.DataFrame(data)

    df_corr = df[df['issue_ratio'] != float('inf')].copy()
    
    if len(df_corr) > 1:
        corr_contributors_ratio = df_corr['contributors'].corr(df_corr['issue_ratio'])
        print(f"\nКорреляция количества контрибьютеров и соотношения открытых/закрытых задач: {corr_contributors_ratio:.3f}")
        
        corr_contributors_open = df_corr['contributors'].corr(df_corr['open_issues'])
        print(f"Корреляция количества контрибьютеров и количества открытых задач: {corr_contributors_open:.3f}")
        
        corr_contributors_closed = df_corr['contributors'].corr(df_corr['closed_issues'])
        print(f"Корреляция количества контрибьютеров и количества закрытых задач: {corr_contributors_closed:.3f}")
    